# 线性回归从零实现

## 导包

In [63]:
import random
import torch
print(torch.__version__)

1.12.0+cu113


## 构造数据集
使用线性模型参数$\mathbf{w} = [2, -3.4]^\top$、$b = 4.2$
和噪声项$\epsilon$生成数据集及其标签:
$$\mathbf{y}= \mathbf{X} \mathbf{w} + b + \mathbf\epsilon.$$

In [64]:
def synthetic_data(w, b, num_examples):  
    """生成y=Xw+b+噪声"""
    X = torch.normal(0, 1, (num_examples,len(w))) #X的列和w的长度相同
    y = torch.matmul(X, w) + b
    y += torch.normal(0, 1, y.size()) #噪声
    return X, y.reshape((-1, 1)) #返回x和一维向量y

In [65]:
#测试
test1 = torch.normal(0, 1, (1,3))
test1

tensor([[-1.5697,  1.3418, -0.7302]])

形状验证

In [66]:
num_examples = 100
w = torch.tensor([2,-3.4], dtype=torch.float32)  
b = torch.tensor(4.2, dtype=torch.float32)

features, labels = synthetic_data(w, b, num_examples)  #w被广播为(1,3)然后与X相乘
print(f"feature[0]: {features[0]}", features.shape, "\n", f"labels[0]: {labels[0]}", labels.shape)


feature[0]: tensor([ 1.0994, -0.1303]) torch.Size([100, 2]) 
 labels[0]: tensor([7.4674]) torch.Size([100, 1])


## 读取数据集
该函数能打乱数据集中的样本并以小批量方式获取数据




In [67]:
def data_iter(batch_size, features, labels):
    num_features = len(features)
    index_features = list(range(num_features))
    # 打乱样本，随机读取
    random.shuffle(index_features)
    #小批次读取
    for i in range(0, num_features, batch_size):
        index_features = torch.tensor(index_features[i:min(i + batch_size, num_features)]) #索引不要搞错
        yield features[index_features], labels[index_features]



采用小批量，因此函数执行一次就采样一小批样本

In [68]:
batch_size = 10

for X, y in data_iter(batch_size, features, labels):
    print(X, '\n', y)
    break #取第一个batch

    

tensor([[ 0.0120,  2.6128],
        [ 0.0300,  0.4125],
        [ 0.6042,  0.1821],
        [ 1.7538,  0.0587],
        [ 0.0518, -0.5793],
        [ 0.9172,  0.4526],
        [-1.4808, -0.3935],
        [ 0.6656,  0.3617],
        [-1.3235, -0.3731],
        [-0.3174,  0.8685]]) 
 tensor([[-5.1295],
        [ 2.4192],
        [ 3.0320],
        [ 7.4166],
        [ 5.1424],
        [ 5.3774],
        [ 1.8414],
        [ 4.7889],
        [ 2.5387],
        [ 2.7126]])


## 初始化模型参数，定义模型、损失函数以及优化算法

通过从均值为0、标准差为0.01的正态分布中采样随机数来初始化权重，
并将偏置初始化为0。

优化算法是sgd：小批量随机梯度下降更新参数

In [69]:
#初始化模型参数
W = torch.normal(0, 0.01, size=(2,1), requires_grad=True)
b = torch.zeros(1, requires_grad=True)

#模型
def linreg(X, W, b):
    return torch.matmul(X, W) + b

#损失函数
def squared_loss(y_hat, y):
    loss = (y_hat - y) **2 / 2
    return loss

#优化算法
def sgd(params, lr, batch_size):
    with torch.no_grad():  #更新参数值时要禁用梯度计算
        for param in params:
            param -= lr * param.grad / batch_size
            param.grad.zero_()  #梯度清零

## 训练
先算loss、再反向传播求梯度，最后用sgd更新参数，并打印epoch和loss

In [70]:
lr = 0.01
batch_size = 10

for epoch in range(5):
    for X, y in data_iter(batch_size, features, labels):
        y_hat = linreg(X, W, b)
        loss = squared_loss(y_hat, y)
        loss.sum().backward()
        sgd([W,b], lr, batch_size)
    with torch.no_grad(): 
        train_loss = squared_loss(linreg(X, W, b), train_loss)  #注意这里算train_loss
        print(f'epoch: {epoch} ', f'train_loss: {loss}')
        loss = 0

epoch: 0  train_loss: tensor([], size=(0, 1), grad_fn=<DivBackward0>)
epoch: 1  train_loss: tensor([], size=(0, 1), grad_fn=<DivBackward0>)
epoch: 2  train_loss: tensor([], size=(0, 1), grad_fn=<DivBackward0>)
epoch: 3  train_loss: tensor([], size=(0, 1), grad_fn=<DivBackward0>)
epoch: 4  train_loss: tensor([], size=(0, 1), grad_fn=<DivBackward0>)


C:\Users\aozhi\AppData\Local\Temp\ipykernel_12124\2053234082.py:8: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  index_features = torch.tensor(index_features[i:min(i + batch_size, num_features)]) #索引不要搞错
